In [1]:
# Bibliotecas usadas para ler os dados, treinar e avaliar as redes
import os
import json
import joblib  # Para salvar o nosso normalizador (scaler)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm  # Barra de progresso

# Métricas do sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Usa uma seed fixa para facilitar a repetição dos resultados
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Conecta o treinamento com a placa de vídeo
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo usado: {DEVICE}")


Dispositivo usado: cuda


In [2]:
# Lê os melhores parâmetros encontrados pelo notebook de tuning
DIR_MODELOS = "modelos_treinados"
os.makedirs(DIR_MODELOS, exist_ok=True)

# Abre o arquivo de resumo do Optuna
with open("tuning_bayesiano/resumo_melhores_hp_multiclasse.json", "r") as f:
    melhores_hp = json.load(f)

In [3]:
# Prepara os dados e separa um teste que fica de fora do treinamento
CENARIO = "multiclasse"

# Leitura do dataset
df = pd.read_csv("Normal+Attacks.csv")

# X contém as entradas numéricas; y guarda a classe que queremos prever
X = df.drop(columns=["label"], errors="ignore")
X = X.select_dtypes(include="number")

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df["label"])
n_classes = len(le.classes_)

joblib.dump(le, "modelos_treinados/label_encoder_multiclasse.joblib")

# Divisão em Treino e Teste (usando os 25% reais para testar a matriz de confusão)
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

# Ajusta a escala usando só o treino e aplica a mesma transformação no teste
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Salva o scaler em disco para usar nos ataques adversariais depois
import joblib
os.makedirs("modelos_treinados", exist_ok=True)
joblib.dump(scaler, "modelos_treinados/scaler_multiclasse.joblib")

# Converte os arrays para tensores, que são o formato usado pelo PyTorch
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_raw, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_raw, dtype=torch.long)


# Cria o DataLoader do conjunto de teste completo
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

N_FEATURES = X_train_scaled.shape[1]
print(f"\nPipeline de dados concluído! Scaler e Encoder salvos em 'modelos_treinados'.")
print(f"Número de colunas de entrada (N_FEATURES): {N_FEATURES}")
print(f"Número de classes identificadas no drone: {n_classes} ({list(le.classes_)})")
print(f"Registros para o Treino: {X_train_tensor.shape[0]}")



Pipeline de dados concluído! Scaler e Encoder salvos em 'modelos_treinados'.
Número de colunas de entrada (N_FEATURES): 44
Número de classes identificadas no drone: 10 (['Bruteforce', 'DDoS', 'DoS', 'Evil', 'FakeLanding', 'MITM', 'Normal', 'Reco', 'Reply', 'Scanning'])
Registros para o Treino: 112075


In [4]:
# Cada classe monta uma arquitetura usando os parâmetros salvos pelo tuning.

class MLP(nn.Module):
    def __init__(self, input_dim, config, n_classes):
        super().__init__()
        camadas = []
        prev = input_dim
        n_camadas = config["hiperparametros"]["n_camadas"]
        ativacao = config["hiperparametros"]["ativacao"]
        
        for i in range(n_camadas):
            unidades = config["hiperparametros"][f"unidades_{i}"]
            dropout = config["hiperparametros"][f"dropout_{i}"]
            camadas.append(nn.Linear(prev, unidades))
            camadas.append(nn.ReLU() if ativacao == "relu" else nn.Tanh())
            if dropout > 0: camadas.append(nn.Dropout(dropout))
            prev = unidades
            
        camadas.append(nn.Linear(prev, n_classes))
        self.rede = nn.Sequential(*camadas)

    def forward(self, x): return self.rede(x)

# RNN, GRU e LSTM leem as colunas como uma sequência.
class Recorrente(nn.Module):
    def __init__(self, tipo_celula, input_dim, config, n_classes):
        super().__init__()
        self.tipo_celula = tipo_celula
        self.camadas = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        n_camadas = config["hiperparametros"]["n_camadas"]
        prev = input_dim
        
        for i in range(n_camadas):
            unidades = config["hiperparametros"][f"unidades_{i}"]
            dropout = config["hiperparametros"][f"dropout_{i}"]
            if tipo_celula == "RNN": camada = nn.RNN(prev, unidades, batch_first=True)
            elif tipo_celula == "GRU": camada = nn.GRU(prev, unidades, batch_first=True)
            else: camada = nn.LSTM(prev, unidades, batch_first=True)
            self.camadas.append(camada)
            self.dropouts.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
            prev = unidades
            
        self.densa = nn.Linear(prev, 64)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(64, n_classes)

    def forward(self, x):
        for i, (camada, drop) in enumerate(zip(self.camadas, self.dropouts)):
            out, _ = camada(x)
            out = drop(out)
            if i < len(self.camadas) - 1: x = out
            else: x = out[:, -1, :]
        return self.saida(self.relu(self.densa(x)))

# A CNN também aplica filtros ao longo das colunas de entrada.
class CNN(nn.Module):
    def __init__(self, input_channels, config, n_classes):
        super().__init__()
        camadas = []
        prev = input_channels
        
        n_blocos = config["hiperparametros"]["n_blocos_conv"]
        kernel_size = config["hiperparametros"]["kernel_size"]
        
        for i in range(n_blocos):
            filtros = config["hiperparametros"][f"filtros_{i}"]
            camadas.append(nn.Conv1d(prev, filtros, kernel_size=kernel_size, padding=kernel_size//2))
            camadas.append(nn.ReLU())
            camadas.append(nn.MaxPool1d(kernel_size=2, ceil_mode=True))
            prev = filtros
            
        self.conv = nn.Sequential(*camadas)
        self.pool_global = nn.AdaptiveAvgPool1d(1)
        self.densa = nn.Linear(prev, 128)
        self.relu = nn.ReLU()
        
        self.dropout = nn.Dropout(config["hiperparametros"].get("dropout_final", 0.2))
        
        self.saida = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.conv(x)
        x = self.pool_global(x).squeeze(-1)
        return self.saida(self.dropout(self.relu(self.densa(x))))

print("Classes das redes neurais carregadas com sucesso!")


Classes das redes neurais carregadas com sucesso!


In [5]:
# Separa os dados em lotes e define o treino final de cada modelo
BATCH_SIZE = 256
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

EPOCHS = 50

def treinar_modelo_definitivo(nome, modelo_instancia, lr):
    print("\n" + "=" * 60)
    print(f"TREINAMENTO: {nome}")
    print("=" * 60)
    
    modelo_instancia = modelo_instancia.to(DEVICE)
    # A rede devolve uma nota por classe; o rótulo indica a classe correta.
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(modelo_instancia.parameters(), lr=lr)
    
    for epoch in range(EPOCHS):
        modelo_instancia.train()
        # Mostra o andamento do treino dessa época com o tqdm.
        loop_treino = tqdm(train_loader, desc=f"Época {epoch+1}/{EPOCHS}", leave=False)
        
        for xb, yb in loop_treino:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if nome in ["RNN", "GRU", "LSTM"]: xb = xb.unsqueeze(-1)
            elif nome == "CNN": xb = xb.unsqueeze(-1).permute(0, 2, 1)
            
            optimizer.zero_grad()
            loss = criterion(modelo_instancia(xb), yb)
            loss.backward()
            optimizer.step()
            loop_treino.set_postfix(loss=f"{loss.item():.4f}")
            
    # Salva os pesos e informações usadas depois para reconstruir a rede.
    caminho_salvamento = f"{DIR_MODELOS}/{nome}_multiclasse.pt"
    torch.save({
        "state_dict": modelo_instancia.state_dict(),
        "n_classes": n_classes,
        "n_features": N_FEATURES
    }, caminho_salvamento)
    print(f"Cérebro neural {nome} salvo com sucesso em: {caminho_salvamento}")
    return modelo_instancia

# Monta cada rede com os parâmetros salvos pelo Optuna
instancias_modelos = {
    "MLP":  MLP(N_FEATURES, melhores_hp["MLP"], n_classes),
    "RNN":  Recorrente("RNN", 1, melhores_hp["RNN"], n_classes),
    "CNN":  CNN(1, melhores_hp["CNN"], n_classes),
    "GRU":  Recorrente("GRU", 1, melhores_hp["GRU"], n_classes),
    "LSTM": Recorrente("LSTM", 1, melhores_hp["LSTM"], n_classes)
}

# Treina e guarda cada arquitetura
modelos_treinados = {}
for nome in ["MLP", "RNN", "CNN", "GRU", "LSTM"]:
    lr_modelo = melhores_hp[nome]["hiperparametros"]["learning_rate"]
    modelos_treinados[nome] = treinar_modelo_definitivo(nome, instancias_modelos[nome], lr_modelo)



TREINAMENTO: MLP


Cérebro neural MLP salvo com sucesso em: modelos_treinados/MLP_multiclasse.pt

TREINAMENTO: RNN


Cérebro neural RNN salvo com sucesso em: modelos_treinados/RNN_multiclasse.pt

TREINAMENTO: CNN


Cérebro neural CNN salvo com sucesso em: modelos_treinados/CNN_multiclasse.pt

TREINAMENTO: GRU


Cérebro neural GRU salvo com sucesso em: modelos_treinados/GRU_multiclasse.pt

TREINAMENTO: LSTM


Cérebro neural LSTM salvo com sucesso em: modelos_treinados/LSTM_multiclasse.pt


In [6]:
# Avalia no teste e salva uma matriz de confusão para cada modelo

# Garante que uma pasta para os gráficos gerados exista no projeto e cria se não existir
DIR_GRAFICOS = "../graficos"
os.makedirs(DIR_GRAFICOS, exist_ok=True)

classes_nomes = list(le.classes_)

for nome, modelo in modelos_treinados.items():
    modelo.eval()
    predicoes_completas = []
    
    # Faz previsões em lotes para não carregar o teste inteiro na GPU e evitar estourar a memória dela
    with torch.no_grad():
        for xb_t, _ in test_loader:
            xb_t = xb_t.to(DEVICE)
            if nome in ["RNN", "GRU", "LSTM"]: 
                xb_t = xb_t.unsqueeze(-1)
            elif nome == "CNN": 
                xb_t = xb_t.unsqueeze(-1).permute(0, 2, 1)
            
            logits = modelo(xb_t)

            # A classe prevista é a que recebeu a maior saída da rede.
            preds_indices = torch.argmax(logits, dim=1).cpu().numpy()
            predicoes_completas.append(preds_indices)
            
    y_pred = np.concatenate(predicoes_completas)
    y_true = np.asarray(y_test_raw) 
    
    # Emissão dos relatórios no terminal
    print(f"\nRELATÓRIO ESTATÍSTICO DE DESEMPENHO MULTICLASSE: {nome}")
    print(classification_report(y_true, y_pred, target_names=classes_nomes))
    
    # Cálculo matemático da Matriz de Confusão
    cm = confusion_matrix(y_true, y_pred)
    
    # Cada linha representa uma classe real, o que ajuda a comparar classes com tamanhos diferentes.
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Montagem da matriz de confusão usando o Seaborn e Matplotlib
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm_norm, 
        annot=True,          # Adiciona os números em texto dentro dos quadrados
        fmt=".2f",           # Limita a exibição a duas casas decimais
        cmap="Oranges",        # Pra diferenciar do binário
        xticklabels=classes_nomes, 
        yticklabels=classes_nomes
    )
    
    plt.title(f"Matriz de Confusão - {nome} Multiclasse")
    plt.xlabel("Previsão do Modelo (%)")
    plt.ylabel("Classe Real (%)")
    plt.tight_layout()
    
    # Salva cada gráfico de matriz no pc
    caminho_fig = os.path.join(DIR_GRAFICOS, f"classificacao_multiclasse_original_matriz_{nome}.png")
    plt.savefig(caminho_fig, dpi=150)
    plt.close()
    print(f"Gráfico de matriz exportado para: {caminho_fig}")



RELATÓRIO ESTATÍSTICO DE DESEMPENHO MULTICLASSE: MLP
              precision    recall  f1-score   support

  Bruteforce       0.99      1.00      0.99      1276
        DDoS       1.00      1.00      1.00      3573
         DoS       1.00      1.00      1.00      3530
        Evil       0.00      0.00      0.00        12
 FakeLanding       0.00      0.00      0.00        50
        MITM       0.62      0.92      0.74       291
      Normal       0.85      0.23      0.36      3361
        Reco       0.00      0.00      0.00     12534
       Reply       0.84      0.41      0.55       198
    Scanning       0.45      1.00      0.62     12534

    accuracy                           0.59     37359
   macro avg       0.58      0.56      0.53     37359
weighted avg       0.46      0.59      0.47     37359



/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

Gráfico de matriz exportado para: ../graficos/classificacao_multiclasse_original_matriz_MLP.png

RELATÓRIO ESTATÍSTICO DE DESEMPENHO MULTICLASSE: RNN


/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

              precision    recall  f1-score   support

  Bruteforce       0.99      0.99      0.99      1276
        DDoS       1.00      1.00      1.00      3573
         DoS       1.00      1.00      1.00      3530
        Evil       0.00      0.00      0.00        12
 FakeLanding       0.00      0.00      0.00        50
        MITM       0.60      0.91      0.73       291
      Normal       0.82      0.23      0.36      3361
        Reco       0.45      1.00      0.62     12534
       Reply       0.83      0.29      0.43       198
    Scanning       0.00      0.00      0.00     12534

    accuracy                           0.59     37359
   macro avg       0.57      0.54      0.51     37359
weighted avg       0.46      0.59      0.47     37359

Gráfico de matriz exportado para: ../graficos/classificacao_multiclasse_original_matriz_RNN.png

RELATÓRIO ESTATÍSTICO DE DESEMPENHO MULTICLASSE: CNN


...

              precision    recall  f1-score   support

  Bruteforce       0.99      1.00      0.99      1276
        DDoS       1.00      1.00      1.00      3573
         DoS       1.00      1.00      1.00      3530
        Evil       0.00      0.00      0.00        12
 FakeLanding       1.00      0.06      0.11        50
        MITM       0.62      0.92      0.74       291
      Normal       0.86      0.23      0.36      3361
        Reco       0.42      0.02      0.04     12534
       Reply       0.85      0.41      0.55       198
    Scanning       0.45      0.97      0.62     12534

    accuracy                           0.59     37359
   macro avg       0.72      0.56      0.54     37359
weighted avg       0.60      0.59      0.49     37359

Gráfico de matriz exportado para: ../graficos/classificacao_multiclasse_original_matriz_CNN.png

RELATÓRIO ESTATÍSTICO DE DESEMPENHO MULTICLASSE: GRU
              precision    recall  f1-score   support

  Bruteforce       0.99      1.00   

/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

Gráfico de matriz exportado para: ../graficos/classificacao_multiclasse_original_matriz_GRU.png

RELATÓRIO ESTATÍSTICO DE DESEMPENHO MULTICLASSE: LSTM
              precision    recall  f1-score   support

  Bruteforce       0.99      1.00      0.99      1276
        DDoS       1.00      1.00      1.00      3573
         DoS       1.00      1.00      1.00      3530
        Evil       0.00      0.00      0.00        12
 FakeLanding       0.00      0.00      0.00        50
        MITM       0.56      0.96      0.71       291
      Normal       0.81      0.22      0.34      3361
        Reco       0.42      0.31      0.35     12534
       Reply       0.60      0.06      0.11       198
    Scanning       0.47      0.69      0.56     12534

    accuracy                           0.59     37359
   macro avg       0.58      0.52      0.51     37359
weighted avg       0.60      0.59      0.57     37359



/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/luish/VSCode/IC/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", res

Gráfico de matriz exportado para: ../graficos/classificacao_multiclasse_original_matriz_LSTM.png


Com esses resultados, e dando uma breve analisada no dataset, podemos ver que há um desbalanceamento enorme: das ~150 mil linhas, cerca de 100 mil são compostas somente de ataques Reco e Scanning, que são, essencialmente, a mesma coisa. Então, vamos fundir essas 2 classes, e remover algumas redundâncias no dataset também.

In [7]:
# Classificacao multiclasse com dados tratados: une Reco/Scanning e avalia com divisao agrupada, preservando os resultados originais.
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score

# Mantem a reproducibilidade e cria uma pasta exclusiva para esta execucao.
SEED_TRATADO = 42
BATCH_SIZE_TRATADO = 256
EPOCHS_TRATADO = 50
DEVICE_TRATADO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(SEED_TRATADO)
torch.manual_seed(SEED_TRATADO)

# Esta pasta guarda somente os resultados atuais; uma nova execucao substitui os arquivos desta etapa.
VERSAO_DADOS = "dados_tratados"
DIR_TRATADO = "classificacao_multiclasse_tratada"
DIR_MODELOS_TRATADO = os.path.join(DIR_TRATADO, "modelos")
DIR_GRAFICOS_TRATADO = os.path.join("..", "graficos")
os.makedirs(DIR_MODELOS_TRATADO, exist_ok=True)
os.makedirs(DIR_GRAFICOS_TRATADO, exist_ok=True)

# Reutiliza os hiperparametros ja encontrados no tuning multiclasse.
with open("tuning_bayesiano/resumo_melhores_hp_multiclasse.json", "r") as arquivo:
    melhores_hp_tratado = json.load(arquivo)

df_original_exp = pd.read_csv("Normal+Attacks.csv")
linhas_originais_exp = len(df_original_exp)
contagem_original_exp = df_original_exp["label"].value_counts().to_dict()
df_tratado = df_original_exp.copy()

# Faz a fusao antes da deduplicacao para colapsar registros iguais que diferiam apenas entre Reco e Scanning.
df_tratado["label"] = df_tratado["label"].replace({
    "Reco": "Reco+Scanning",
    "Scanning": "Reco+Scanning",
})

# Confirma que a feature e realmente constante antes de remove-la.
coluna_constante_exp = "fwd_seg_size_min"
if df_tratado[coluna_constante_exp].nunique(dropna=False) != 1:
    raise ValueError(f"{coluna_constante_exp} nao e constante")
df_tratado = df_tratado.drop(columns=[coluna_constante_exp])

X_todas_exp = df_tratado.drop(columns=["label"]).select_dtypes(include="number")
feature_names_exp = X_todas_exp.columns.tolist()
dados_exp = X_todas_exp.copy()
dados_exp["target"] = df_tratado["label"].to_numpy()

# Deduplica o par features+classe: vetores identicos com classes diferentes continuam identificaveis.
dados_exp = dados_exp.drop_duplicates().reset_index(drop=True)
linhas_apos_limpeza_exp = len(dados_exp)
X_tratado = dados_exp[feature_names_exp]
le_tratado = LabelEncoder()
y_tratado = le_tratado.fit_transform(dados_exp["target"])
n_classes_tratado = len(le_tratado.classes_)

# Mantém cada vetor de features no mesmo lado do split, mesmo quando houver rótulos conflitantes.
# Usa somente o primeiro fold de aproximadamente 25%; não executa validação cruzada completa.
grupos_exp = pd.util.hash_pandas_object(X_tratado, index=False).to_numpy()
splitter_exp = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED_TRATADO)
indices_treino_exp, indices_teste_exp = next(
    splitter_exp.split(X_tratado, y_tratado, groups=grupos_exp)
)
X_train_raw_exp = X_tratado.iloc[indices_treino_exp]
X_test_raw_exp = X_tratado.iloc[indices_teste_exp]
y_train_exp = y_tratado[indices_treino_exp]
y_test_exp = y_tratado[indices_teste_exp]

# Ajusta a normalizacao somente no treino para nao deixar estatisticas do teste vazarem para o modelo.
scaler_tratado = StandardScaler()
X_train_exp = scaler_tratado.fit_transform(X_train_raw_exp)
X_test_exp = scaler_tratado.transform(X_test_raw_exp)
N_FEATURES_TRATADO = X_train_exp.shape[1]
X_train_tensor_exp = torch.tensor(X_train_exp, dtype=torch.float32)
X_test_tensor_exp = torch.tensor(X_test_exp, dtype=torch.float32)
y_train_tensor_exp = torch.tensor(y_train_exp, dtype=torch.long)
y_test_tensor_exp = torch.tensor(y_test_exp, dtype=torch.long)
train_loader_exp = DataLoader(
    TensorDataset(X_train_tensor_exp, y_train_tensor_exp),
    batch_size=BATCH_SIZE_TRATADO,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED_TRATADO),
)
test_loader_exp = DataLoader(
    TensorDataset(X_test_tensor_exp, y_test_tensor_exp), batch_size=1024, shuffle=False
)

class MLPTratado(nn.Module):
    def __init__(self, input_dim, config, n_classes):
        super().__init__()
        hp = config["hiperparametros"]
        camadas = []
        anterior = input_dim
        for indice in range(hp["n_camadas"]):
            unidades = hp[f"unidades_{indice}"]
            dropout = hp[f"dropout_{indice}"]
            camadas.extend([nn.Linear(anterior, unidades), nn.ReLU() if hp["ativacao"] == "relu" else nn.Tanh()])
            if dropout > 0:
                camadas.append(nn.Dropout(dropout))
            anterior = unidades
        camadas.append(nn.Linear(anterior, n_classes))
        self.rede = nn.Sequential(*camadas)

    def forward(self, x):
        return self.rede(x)

class RecorrenteTratado(nn.Module):
    def __init__(self, tipo, config, n_classes):
        super().__init__()
        hp = config["hiperparametros"]
        self.camadas = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        anterior = 1
        for indice in range(hp["n_camadas"]):
            unidades = hp[f"unidades_{indice}"]
            dropout = hp[f"dropout_{indice}"]
            self.camadas.append({"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[tipo](anterior, unidades, batch_first=True))
            self.dropouts.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
            anterior = unidades
        self.densa = nn.Linear(anterior, 64)
        self.saida = nn.Linear(64, n_classes)

    def forward(self, x):
        for indice, (camada, dropout) in enumerate(zip(self.camadas, self.dropouts)):
            saida, _ = camada(x)
            saida = dropout(saida)
            x = saida if indice < len(self.camadas) - 1 else saida[:, -1, :]
        return self.saida(torch.relu(self.densa(x)))

class CNNTratado(nn.Module):
    def __init__(self, config, n_classes):
        super().__init__()
        hp = config["hiperparametros"]
        camadas = []
        anterior = 1
        kernel_size = hp["kernel_size"]
        for indice in range(hp["n_blocos_conv"]):
            filtros = hp[f"filtros_{indice}"]
            camadas.extend([nn.Conv1d(anterior, filtros, kernel_size, padding=kernel_size // 2), nn.ReLU(), nn.MaxPool1d(2, ceil_mode=True)])
            anterior = filtros
        self.conv = nn.Sequential(*camadas)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.densa = nn.Linear(anterior, 128)
        self.dropout = nn.Dropout(hp.get("dropout_final", 0.2))
        self.saida = nn.Linear(128, n_classes)

    def forward(self, x):
        x = self.pool(self.conv(x)).squeeze(-1)
        return self.saida(self.dropout(torch.relu(self.densa(x))))


# Prepara o formato esperado pelas arquiteturas que interpretam as colunas como uma sequencia.
def preparar_entrada_tratado(nome, xb):
    if nome in ("RNN", "GRU", "LSTM"):
        return xb.unsqueeze(-1)
    if nome == "CNN":
        return xb.unsqueeze(-1).permute(0, 2, 1)
    return xb


construtores_exp = {
    "MLP": lambda: MLPTratado(N_FEATURES_TRATADO, melhores_hp_tratado["MLP"], n_classes_tratado),
    "RNN": lambda: RecorrenteTratado("RNN", melhores_hp_tratado["RNN"], n_classes_tratado),
    "CNN": lambda: CNNTratado(melhores_hp_tratado["CNN"], n_classes_tratado),
    "GRU": lambda: RecorrenteTratado("GRU", melhores_hp_tratado["GRU"], n_classes_tratado),
    "LSTM": lambda: RecorrenteTratado("LSTM", melhores_hp_tratado["LSTM"], n_classes_tratado),
}

# Salva junto do modelo os objetos necessarios para repetir o mesmo pre-processamento na inferencia.
joblib.dump(le_tratado, os.path.join(DIR_MODELOS_TRATADO, "label_encoder.joblib"))
joblib.dump(scaler_tratado, os.path.join(DIR_MODELOS_TRATADO, "scaler.joblib"))
metricas_treinamento = {}

for nome, construir_modelo in construtores_exp.items():
    print(f"\nTreinando {nome} com dados tratados e grupos (multiclasse) ({DEVICE_TRATADO})")
    modelo = construir_modelo().to(DEVICE_TRATADO)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(modelo.parameters(), lr=melhores_hp_tratado[nome]["hiperparametros"]["learning_rate"])

    for epoca in range(EPOCHS_TRATADO):
        modelo.train()
        for xb, yb in tqdm(train_loader_exp, desc=f"{nome} {epoca + 1}/{EPOCHS_TRATADO}", leave=False):
            xb, yb = xb.to(DEVICE_TRATADO), yb.to(DEVICE_TRATADO)
            xb = preparar_entrada_tratado(nome, xb)
            optimizer.zero_grad()
            loss = criterion(modelo(xb), yb)
            loss.backward()
            optimizer.step()

    modelo.eval()
    predicoes = []
    with torch.no_grad():
        for xb, _ in test_loader_exp:
            xb = preparar_entrada_tratado(nome, xb.to(DEVICE_TRATADO))
            predicoes.extend(torch.argmax(modelo(xb), dim=1).cpu().numpy().tolist())

    y_pred_exp = np.asarray(predicoes)
    relatorio_exp = classification_report(
        y_test_exp,
        y_pred_exp,
        labels=np.arange(n_classes_tratado),
        target_names=le_tratado.classes_.tolist(),
        output_dict=True,
        zero_division=0,
    )
    # A acuracia pesa mais as classes grandes; o F1 macro atribui peso igual a cada classe.
    metricas_treinamento[nome] = {
        "accuracy": float(accuracy_score(y_test_exp, y_pred_exp)),
        "macro_f1": float(f1_score(y_test_exp, y_pred_exp, labels=np.arange(n_classes_tratado), average="macro", zero_division=0)),
        "classification_report": relatorio_exp,
    }
    print(classification_report(
        y_test_exp,
        y_pred_exp,
        labels=np.arange(n_classes_tratado),
        target_names=le_tratado.classes_.tolist(),
        zero_division=0,
    ))

    # Cada arquitetura e salva na pasta desta execucao, nesta pasta de resultados atualizada a cada execucao.
    caminho_pesos = os.path.join(DIR_MODELOS_TRATADO, f"{nome}_multiclasse.pt")
    torch.save({
        "state_dict": modelo.state_dict(),
        "n_classes": n_classes_tratado,
        "n_features": N_FEATURES_TRATADO,
        "class_names": le_tratado.classes_.tolist(),
        "feature_names": feature_names_exp,
    }, caminho_pesos)

    # Normaliza cada linha da matriz para comparar o recall entre classes de tamanhos diferentes.
    matriz_exp = confusion_matrix(y_test_exp, y_pred_exp, labels=np.arange(n_classes_tratado))
    matriz_normalizada_exp = matriz_exp.astype(float) / np.maximum(matriz_exp.sum(axis=1, keepdims=True), 1) * 100
    plt.figure(figsize=(9, 7))
    sns.heatmap(matriz_normalizada_exp, annot=True, fmt=".1f", cmap="Oranges", xticklabels=le_tratado.classes_, yticklabels=le_tratado.classes_)
    plt.title(f"Matriz de Confusao - {nome} Multiclasse (Reco + Scanning, split agrupado)")
    plt.xlabel("Classe prevista (%)")
    plt.ylabel("Classe real (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_GRAFICOS_TRATADO, f"classificacao_multiclasse_tratada_matriz_{nome}.png"), dpi=150)
    plt.close()

# Guarda os relatorios completos, o resumo da limpeza, a composicao do split e as metricas agregadas.
with open(os.path.join(DIR_TRATADO, "metricas_treinamento.json"), "w", encoding="utf-8") as arquivo:
    json.dump(metricas_treinamento, arquivo, indent=2, ensure_ascii=False)

contagens_tratadas = pd.Series(le_tratado.inverse_transform(y_tratado)).value_counts().to_dict()
resumo_treinamento = {
    "versao": VERSAO_DADOS,
    "seed": SEED_TRATADO,
    "epocas": EPOCHS_TRATADO,
    "divisao": "StratifiedGroupKFold: fold 1 de 4; grupos por vetor de features",
    "vetores_de_features_com_rotulos_conflitantes": int((dados_exp.groupby(feature_names_exp, dropna=False)["target"].nunique() > 1).sum()),
    "overlap_grupos_treino_teste": int(len(set(grupos_exp[indices_treino_exp]) & set(grupos_exp[indices_teste_exp]))),
    "linhas_originais": linhas_originais_exp,
    "duplicatas_removidas_apos_fusao_e_remocao_da_coluna": linhas_originais_exp - linhas_apos_limpeza_exp,
    "linhas_apos_limpeza": linhas_apos_limpeza_exp,
    "coluna_removida": coluna_constante_exp,
    "contagem_classes_original": {str(k): int(v) for k, v in contagem_original_exp.items()},
    "contagem_classes_apos_limpeza": {str(k): int(v) for k, v in contagens_tratadas.items()},
    "contagem_teste": {str(le_tratado.inverse_transform([int(k)])[0]): int(v) for k, v in pd.Series(y_test_exp).value_counts().items()},
    "classes_modelo": le_tratado.classes_.tolist(),
    "features": feature_names_exp,
    "linhas_treino": int(len(y_train_exp)),
    "linhas_teste": int(len(y_test_exp)),
    "metricas": {nome: {"accuracy": valores["accuracy"], "macro_f1": valores["macro_f1"]} for nome, valores in metricas_treinamento.items()},
}
with open(os.path.join(DIR_TRATADO, "resumo_treinamento.json"), "w", encoding="utf-8") as arquivo:
    json.dump(resumo_treinamento, arquivo, indent=2, ensure_ascii=False)

print(f"\nClassificacao multiclasse concluida. Resultados atuais em: {DIR_TRATADO}")


Treinando MLP com dados tratados e grupos (multiclasse) (cuda)


               precision    recall  f1-score   support

   Bruteforce       0.99      1.00      1.00      1274
         DDoS       1.00      1.00      1.00      3573
          DoS       1.00      1.00      1.00      3531
         Evil       0.00      0.00      0.00         2
  FakeLanding       0.00      0.00      0.00        17
         MITM       0.62      0.90      0.73        52
       Normal       0.52      0.71      0.60       126
Reco+Scanning       0.92      0.75      0.83       288
        Reply       0.67      0.61      0.64        51

     accuracy                           0.98      8914
    macro avg       0.64      0.66      0.64      8914
 weighted avg       0.98      0.98      0.98      8914


Treinando RNN com dados tratados e grupos (multiclasse) (cuda)


               precision    recall  f1-score   support

   Bruteforce       0.99      1.00      0.99      1274
         DDoS       1.00      1.00      1.00      3573
          DoS       1.00      1.00      1.00      3531
         Evil       0.00      0.00      0.00         2
  FakeLanding       0.00      0.00      0.00        17
         MITM       0.61      0.88      0.72        52
       Normal       0.51      0.17      0.25       126
Reco+Scanning       0.79      0.94      0.86       288
        Reply       0.56      0.80      0.66        51

     accuracy                           0.98      8914
    macro avg       0.61      0.64      0.61      8914
 weighted avg       0.98      0.98      0.98      8914


Treinando CNN com dados tratados e grupos (multiclasse) (cuda)


               precision    recall  f1-score   support

   Bruteforce       0.99      1.00      1.00      1274
         DDoS       1.00      1.00      1.00      3573
          DoS       1.00      1.00      1.00      3531
         Evil       0.00      0.00      0.00         2
  FakeLanding       0.00      0.00      0.00        17
         MITM       0.64      0.90      0.75        52
       Normal       0.66      0.15      0.25       126
Reco+Scanning       0.79      0.97      0.87       288
        Reply       0.57      0.78      0.66        51

     accuracy                           0.98      8914
    macro avg       0.63      0.65      0.61      8914
 weighted avg       0.98      0.98      0.98      8914


Treinando GRU com dados tratados e grupos (multiclasse) (cuda)


               precision    recall  f1-score   support

   Bruteforce       0.99      1.00      1.00      1274
         DDoS       1.00      1.00      1.00      3573
          DoS       1.00      1.00      1.00      3531
         Evil       0.00      0.00      0.00         2
  FakeLanding       0.00      0.00      0.00        17
         MITM       0.67      0.85      0.75        52
       Normal       0.56      0.35      0.43       126
Reco+Scanning       0.79      0.97      0.87       288
        Reply       0.63      0.33      0.44        51

     accuracy                           0.98      8914
    macro avg       0.63      0.61      0.61      8914
 weighted avg       0.98      0.98      0.98      8914


Treinando LSTM com dados tratados e grupos (multiclasse) (cuda)


               precision    recall  f1-score   support

   Bruteforce       0.99      1.00      1.00      1274
         DDoS       1.00      1.00      1.00      3573
          DoS       1.00      1.00      1.00      3531
         Evil       0.00      0.00      0.00         2
  FakeLanding       0.00      0.00      0.00        17
         MITM       0.59      0.88      0.71        52
       Normal       0.51      0.30      0.38       126
Reco+Scanning       0.84      0.86      0.85       288
        Reply       0.55      0.88      0.68        51

     accuracy                           0.98      8914
    macro avg       0.61      0.66      0.62      8914
 weighted avg       0.98      0.98      0.98      8914


Classificacao multiclasse concluida. Resultados atuais em: classificacao_multiclasse_tratada
